In [1]:
from treeflow import TreeFlow
from river_interface import convert_arf,convert_arf_reg
from river_interface import convert_amf
from river.forest.aggregated_mondrian_forest import AMFClassifier
from river.forest.adaptive_random_forest import ARFClassifier, ARFRegressor
import polars as pl
from sklearn.model_selection import *
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn import set_config
import missingno as mno
import seaborn as sns
from sklearn.impute import SimpleImputer
import matplotlib.pyplot as plt

In [2]:

treeflow = TreeFlow(port=8001,work_dir="../test")
treeflow._start_server()

###PolSAR###

In [ ]:
import sys
sys.path.insert(0, '../PolSAR')
from distances import ALL_DISTANCES
import rasterio as rs
from node_tests import NodeTestSampler
from polsar_online_rf import OnlinePolSARRandomForest
from polsar_interface import convert_polsar_rf, make_polsar_router, patch_to_rgb
from example import build_reference_pool, sample_training_patches
from air_polsar import load_air_tiles

In [ ]:
rs.open('../test/train_set/AIR-PolarSAR-Seg-1_HH.tiff')

In [ ]:
mat = rs.open('../test/train_set/AIR-PolarSAR-Seg-2_Combined.tiff').read()

In [ ]:
# mat: (4, 512, 512) = [HH, HV, VH, VV] stacked by row
# -> (512, 512, 2, 2) covariance matrix [[HH, HV], [VH, VV]] per pixel
patch = mat.transpose(1, 2, 0).reshape(512, 512, 2, 2)

img = patch_to_rgb(patch,1,99)
plt.imshow(img)

In [ ]:
def loadDS(path = None):
    labels = []
    data = []
    for i in range(1,351):
        tiff_c = rs.open('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_Combined.tiff').read()
        tiff_label = plt.imread('../test/train_set/AIR-PolarSAR-Seg-'+str(i)+'_gt.png')
        labels.append(tiff_label)
        data.append(tiff_c)
    return data, labels

In [ ]:
data , labels = loadDS()

In [ ]:
import numpy as np

In [ ]:
AIR_CLASSES: list[tuple[str, tuple[int, int, int]]] = [
    ("housing",    (255, 255,   0)),
    ("industrial", (255,   0,   0)),
    ("natural",    (  0, 255,   0)),
    ("land_use",   (  0, 255, 255)),
    ("water",      (  0,   0, 255)),
    ("other",      (255, 255, 255)),
]
AIR_CLASS_NAMES = [name for name, _ in AIR_CLASSES]
AIR_IGNORE = 255  # label for unlabeled (black) ground-truth pixels
rng = np.random.default_rng(0)

In [ ]:
images, labels = load_air_tiles("../test/train_set", indices=[1, 55, 100, 299])
pool = build_reference_pool(images, n=200, rng=rng)
patches, patch_labels = sample_training_patches(
    images, labels, n=5000, patch_size=9, rng=rng, ignore_label=AIR_IGNORE)

In [ ]:
sampler = NodeTestSampler(
patch_size=9,
region_size_min=2,
region_size_max=4,
max_offset=3,
reference_pool=build_reference_pool(images, n=200, rng=rng),
distance_names=list(ALL_DISTANCES.keys()),)
batch_size = 200
rf = OnlinePolSARRandomForest(
        sampler=sampler,
        n_trees=5,           # demo settings — paper uses 50 trees, depth 50
        n_candidates=15,
        max_depth=8,
        tau=60,              # samples a node accumulates before it splits
        n_classes=6,
        random_state=42,
    )

In [ ]:
build

##Regression##

In [ ]:
arf_r = ARFRegressor(max_depth=12)

In [ ]:
df_r = pl.read_csv('../test/mortgage_loan_dataset.csv')

In [ ]:
set_config(transform_output='polars')
labels = df_r["Max Loan Amount (USD)"]
data = df_r.drop(["Max Loan Amount (USD)"])


cat_cols = [c for c, dt in zip(data.columns, data.dtypes) if dt == pl.String]
ct = ColumnTransformer([('cats',OrdinalEncoder(),cat_cols)],remainder='passthrough',verbose_feature_names_out=True)
#data = data.to_dummies(columns=cat_cols)
data = ct.fit_transform(data)
X_train, X_test, Y_train, Y_test = train_test_split(data, labels, test_size=0.33)

In [ ]:
index = 1

for (x, y) in zip(X_train.rows(named=True),Y_train):
    arf_r.learn_one(x, y)
    if index % 5000 == 0:
        treeflow.push(convert_arf_reg(arf_r, n_samples=index))
    index += 1

##Multi-Label Classification##

In [ ]:
df_cml = pl.read_csv('../test/train.csv')

In [ ]:
sns.histplot(df_cml['key'])

In [ ]:
mno.matrix(df_cml.to_pandas())

In [ ]:
df_cml.head(1)

In [ ]:
set_config(transform_output='polars')
labels = df_cml['Popularity','Class']
si = SimpleImputer(strategy='mean')
labels = si.fit_transform(labels)
data = df_cml.drop(['Popularity','Class','Track Name'])


#cat_cols = [c for c, dt in zip(data.columns, data.dtypes) if dt == pl.String]
ct = ColumnTransformer([('instr',SimpleImputer(strategy='constant',fill_value=0),['instrumentalness']),('key',SimpleImputer(strategy='most_frequent'),['key']),('cats',OrdinalEncoder(),['Artist Name'])],remainder='passthrough',verbose_feature_names_out=False)
#data = data.to_dummies(columns=cat_cols)
data = ct.fit_transform(data)
X_train, X_test, Y_train, Y_test = train_test_split(data, labels, test_size=0.33)

In [ ]:
i, (X,y) = next(enumerate(zip(X_train.rows(named=True),Y_train.rows(named=True))))

In [ ]:
from river.multioutput import MultiClassEncoder
from river.forest import ARFClassifier
from river_interface import convert_arf_ml
zipped_df = list(zip(X_train.rows(named=True),Y_train.rows(named=True)))
enc = MultiClassEncoder(model=ARFClassifier(n_models=10, seed=42))
index = 0
for i, (x, y) in enumerate(zip(X_train.rows(named=True),Y_train.rows(named=True))): 
   # y = {"AND": 0, "OR": 1, ...}
    enc.learn_one(x, y)
    if i % 500 == 0:
        treeflow.push(convert_arf_ml(enc, n_samples=i + 1, eval_data=zipped_df[index:i]))
        index = i


## Mixed: genre classification + popularity regression (same trees) ##

`MixedISOUPForest` bags iSOUP multi-target trees ARF-style (Poisson resampling). The `Class` group is one-hot-expanded into 0/1 indicator targets internally, `Popularity` stays continuous — so every node carries the genre histogram AND the popularity mean/std, and the frontend group picker switches between pie and gradient rendering.

`reg_scale` matters: iSOUP's split merit averages raw per-target variances, so scale continuous targets to roughly unit range (0–100 popularity → ×0.01) or the class indicators get ignored.

In [ ]:
from river_interface import MixedISOUPForest, convert_isoup_mixed

CLASSES = sorted(df_cml['Class'].drop_nulls().unique().to_list())
mixed_forest = MixedISOUPForest(
    class_groups={'Class': CLASSES},
    reg_targets=['Popularity'],
    reg_scale={'Popularity': 0.01},
    n_models=10, seed=42, grace_period=50,
)

In [ ]:
zipped_mixed = list(zip(X_train.rows(named=True), Y_train.rows(named=True)))

for i, (x, y) in enumerate(zipped_mixed):   # y = {'Popularity': 60.0, 'Class': 5.0}
    mixed_forest.learn_one(x, y)
    if i % 5000 == 0:
        treeflow.push(convert_isoup_mixed(
            mixed_forest, n_samples=i + 1,
            eval_data=zipped_mixed[max(0, i - 200):i],   # sliding window keeps push latency flat
        ))

#Classification#

In [ ]:
#arf = AMFClassifier(n_estimators= 10,max_nodes= 200)
arf_c = ARFClassifier(n_models=10,max_depth=12)

In [ ]:
df_c = pl.read_csv("../test/loan_approval_dataset.csv")

In [ ]:
labels = df_c[" loan_status"]
data = df_c.drop(["loan_id", " loan_status"])
cat_cols = [c for c, dt in zip(data.columns, data.dtypes) if dt == pl.String]
data = data.to_dummies(columns=cat_cols)

X_train, X_test, Y_train, Y_test = train_test_split(data, labels, test_size=0.33)

In [ ]:
index = 1

for (x, y) in zip(X_train.rows(named=True),Y_train):
    arf_c.learn_one(x, y)
    if index % 500 == 0:
        treeflow.push(convert_arf(arf_c, n_samples=index))
    index += 1

In [ ]:
treeflow.shutdown()